In [ ]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import threading
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import numpy as np
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy import stats
import random
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
random.seed(42)

In [ ]:
df=pd.read_csv('/content/D1_UK_Beans_Brand_PPG.csv')
df.fillna(0,inplace=True)

unique_ppg=df.PPG.unique()
unique_ppg

array(['Others', 'Standard Multi', 'Standard Single', 'Small Single',
       'Small Multi', 'Small SNAP POTS'], dtype=object)

In [ ]:
def remove_outliers_iqr(df):

    numeric_cols = df.select_dtypes(include='number').columns
    # numeric_cols=['SalesValue']
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Filter out outliers for the current column
        df = df[~((df[col] < lower_bound) | (df[col] > upper_bound))]

    return df

In [ ]:
# df.columns

In [ ]:
# df.Channel.unique()

In [ ]:
# df=df[df['Channel']=='Supermarkets']
# df=df[df['PPG']=='Standard Multi']

In [ ]:
# df

In [ ]:
# plt.figure(figsize=(8, 6))
# df=df[df['Brand']=='BRANSTON']
# x_value = [i + 1 for i in range(len(df))]
# plt.plot(x_value, df['Cat_Pr_Index'], label='Cat_Pr_Index', marker='o')
# plt.plot(x_value, df['Up_Down_Index'], label='Up_Down_Index', marker='x')

# # Adding title and labels
# plt.title(f'Category price index vs Up down index Values ')
# plt.xlabel('Index')
# plt.ylabel('Price Values')
# plt.xticks(rotation=90)
# plt.legend()
# plt.show()

# Full Data Model Ridge Regression

In [ ]:
def Custom_ridge(df_ch_ppg):
    # Original columns in use
    columns_in_use = ['PPL', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect', 'D1', 'D1_cann',
                      'D1_Comp@Direct', 'D1_Comp@Indirect', 'Seasonality',
                      'Trend', 'Up_Down_Index', 'Cat_Pr_Index']

    # One-hot encode the 'Brand' columnz
    df_encoded = pd.get_dummies(df_ch_ppg, columns=['Brand'], drop_first=False)
    df_encoded = df_encoded.replace({True: 1, False: 0})

    # Get unique brand and encoded column names
    unique_brand = df_ch_ppg['Brand'].unique()
    len_unique_brand = len(unique_brand)
    df_encoded_columns = list(df_encoded.columns)
    encoded_columns = df_encoded_columns[-len_unique_brand:]

    # Interaction terms: Multiply price with each encoded brand column
    for col in encoded_columns:
        df_encoded[f'PPL_{col}'] = df_encoded['PPL'] * df_encoded[col]

    # Columns used for modeling (including interaction terms)
    interaction_columns = [f'PPL_{col}' for col in encoded_columns]
    columns_taken = [item for sublist in [columns_in_use, encoded_columns, interaction_columns] for item in sublist]

    # Target variable
    y_var = df_encoded['Volume']
    columns_in_use=columns_in_use+interaction_columns
    # Standardize only the 'columns_in_use', leave encoded & interaction columns untouched
    scaler = StandardScaler()
    x_var = df_encoded[columns_in_use]  # Only scale columns_in_use
    x_var_scaled = scaler.fit_transform(x_var)

    # Concatenate the scaled columns with the one-hot and interaction columns
    x_full = np.concatenate([x_var_scaled, df_encoded[encoded_columns + interaction_columns].values], axis=1)

    # Train-test split
    x_train, x_test, y_train, y_test = train_test_split(x_full, y_var, test_size=0.2, random_state=42)

    # Adding intercept column to x_train
    x_train_custom_scaled = np.c_[np.ones((x_train.shape[0], 1)), x_train]
    x_test_custom_scaled = np.c_[np.ones((x_test.shape[0], 1)), x_test]
    # Prediction function (including intercept)
    def predict(X, W):
        return X.dot(W[1:]) + W[0]

    # Define the cost function (including L2 regularization)
    def cost_function(params, X, Y, l2_penalty):
        W = params
        Y_pred = predict(X, W)
        cost = np.sum((Y - Y_pred) ** 2) / X.shape[0] + l2_penalty * np.sum(W[1:] ** 2)
        return cost

    # MAPE calculation function
    def mean_absolute_percentage_error(y_true, y_pred):
        y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)
        weighted_mape = 0
        for i in range(len(y_pred)):
            sub = abs(y_true[i] - y_pred[i])
            weighted_mape += sub
        mape_weighted = weighted_mape / np.sum(y_true)
        return mape_weighted * 100

    # Define constraints based on the problem statement
    def get_constraints(feature_names):
        constraints = []
        constraints.append({'type': 'ineq', 'fun': lambda params: -params[1] - 0.01})

        for i, name in enumerate(feature_names):
            if name in ['D1', 'PPL_Comp@Direct', 'PPL_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: params[i]})
            elif name in ['D1_Comp@Direct', 'D1_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: -params[i] - 0.00001})
        return constraints

    # List of L2 penalty values to iterate over
    l2_penalty_list = [0.01, 0.1, 1, 10, 100]

    # Initialize variables to store the best results
    mape_custom_train_prev = float('inf')  # Start with a very large value for comparison
    best_l2_penalty = None
    best_W_opt = None
    best_y_pred_custom_train = None
    best_y_pred_custom_test = None

    for l2_penalty in l2_penalty_list:
        # Initial parameters (weights + intercept)
        initial_params = np.zeros(x_train_custom_scaled.shape[1] + 1)

        # Get the constraints
        constraints = get_constraints(columns_taken)

        # Minimize the cost function using SLSQP
        result = minimize(
            cost_function,
            initial_params,
            args=(x_train_custom_scaled, y_train, l2_penalty),
            constraints=constraints,
            method='SLSQP'
        )

        # Extract optimized parameters
        W_opt = result.x

        # Make predictions on the training set
        y_pred_custom_train = predict(x_train_custom_scaled, W_opt)
        y_pred_custom_test = predict(x_test_custom_scaled, W_opt)

        # Calculate MAPE for the custom implementation
        mape_custom_train_new = mean_absolute_percentage_error(y_train, y_pred_custom_train)

        # Update the best values if the current MAPE is better
        if mape_custom_train_new < mape_custom_train_prev:
            mape_custom_train_prev = mape_custom_train_new
            best_l2_penalty = l2_penalty
            best_W_opt = W_opt
            best_y_pred_custom_train = y_pred_custom_train
            best_y_pred_custom_test = y_pred_custom_test
            mape_custom_train = mape_custom_train_new  # Update train MAPE
            mape_custom_test = mean_absolute_percentage_error(y_test, y_pred_custom_test)  # Update test MAPE

    # Now after the loop, use the best_W_opt to report the final results
    print(f'Best l2_penalty: {best_l2_penalty}')
    print(f'MAPE (Train): {mape_custom_train:.2f}%')
    print(f'MAPE (Test): {mape_custom_test:.2f}%')

    # Optionally, you can plot the results using the best test predictions
    plt.figure(figsize=(16, 6))
    x_value = [i + 1 for i in range(len(y_test))]
    plt.plot(x_value, y_test, label='Actual Values', marker='o')
    plt.plot(x_value, best_y_pred_custom_test, label='Predicted Values by Custom Code', marker='x')

    # Adding title and labels
    plt.title(f'Actual vs Predicted Values (Best l2_penalty={best_l2_penalty})')
    plt.xlabel('Index')
    plt.ylabel('Values')
    plt.xticks(rotation=90)
    plt.legend()
    plt.show()

    return mape_custom_train, mape_custom_test,W_opt


In [ ]:
unique_channel=df.Channel.unique()
unique_channel

array(['Convenience', 'Iceland', 'Supermarkets', 'Tesco'], dtype=object)

In [ ]:

unique_channel=df.Channel.unique()
unique_channel
channel=[]
ppg_list=[]
unique_brand_list=[]
train_mape=[]
test_mape=[]
for ch in unique_channel:
  df_ch=df[df['Channel']==ch]

  unique_ppg=df_ch.PPG.unique()
  unique_ppg
  for ppg in unique_ppg:
    df_ch_ppg=df_ch[df_ch['PPG']==ppg]
    unique_brand=df_ch_ppg.Brand.unique()


    df_ch_ppg=df_ch_ppg[['Market', 'Channel', 'Region', 'Category', 'SubCategory', 'Brand',
          'Variant', 'PackType', 'PPG', 'PackSize', 'Year',
          'Month', 'Week', 'Date', 'SalesValue', 'Volume', 'VolumeUnits', 'D1',
          'PPU', 'PPL', 'Cat_Vol_all_Chn', 'Cat_Vol_in_Chn', 'Seasonality',
          'Trend', 'Cat_Sales_in_Chn', 'Volume_sum_cann', 'D1_cann',
          'D1_Comp@Direct', 'D1_Comp@Indirect', 'PPU_cann', 'PPU_Comp@Direct',
          'PPU_Comp@Indirect', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect',
          'Up_Down_Index', 'Cat_Pr_Index']]

    df_ch_ppg.dropna(inplace=True)
    df_ch_ppg.reset_index(drop=True,inplace=True)
    # df_ch_ppg=remove_outliers_iqr(df_ch_ppg)
    if len(df_ch_ppg)>30:


      print(f'enough data points( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')
      mape_train,mape_test,W_opt=Custom_ridge(df_ch_ppg)
      channel.append(ch)
      ppg_list.append(ppg)
      train_mape.append(mape_train)
      test_mape.append(mape_test)
      unique_brand_list.append(unique_brand)

      print(W_opt)
    else :
      print(f'not enough data points ( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
result_df=pd.DataFrame({'Channel':channel,'PPG':ppg_list,'Brands':unique_brand_list,'Train_MAPE':train_mape,'Test_MAPE':test_mape})

result_df['Train_MAPE']=result_df['Train_MAPE'].round(2)
result_df['Test_MAPE']=result_df['Test_MAPE'].round(2)
result_df

,Channel,PPG,Brands,Train_MAPE,Test_MAPE
0,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",26.28,31.77
1,Convenience,Standard Multi,"[BRANSTON, HEINZ Standard, Private Label Std]",26.57,30.01
2,Convenience,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",23.57,23.43
3,Convenience,Small Single,"[HEINZ Flavoured, HEINZ Standard, Private Labe...",24.82,24.62
4,Convenience,Small Multi,[HEINZ Standard],23.52,29.41
5,Convenience,Small SNAP POTS,[HEINZ Standard],11.44,15.96
6,Iceland,Small Multi,"[BRANSTON, HEINZ Standard]",26.50,18.46
7,Iceland,Small Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",58.75,93.46
8,Iceland,Standard Multi,"[BRANSTON, HEINZ Standard]",12.59,11.37
9,Iceland,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Re...",30.73,35.79


# Non linear

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

In [ ]:
def Custom_ridge_Non_linear(df_ch_ppg):

  columns_in_use=[  'PPL','PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect','D1','D1_cann', 'D1_Comp@Direct','D1_Comp@Indirect',
           'Seasonality', 'Trend', 'Up_Down_Index','Cat_Pr_Index']
  df_encoded = pd.get_dummies(df_ch_ppg, columns=['Brand'], drop_first=False)
  df_encoded = df_encoded.replace({True: 1, False: 0})

  # Get unique brand and encoded column names
  unique_brand = df_ch_ppg.Brand.unique()
  len_unique_brand = len(df_ch_ppg.Brand.unique())
  df_encoded_columns = list(df_encoded.columns)
  encoded_columns = df_encoded_columns[-len_unique_brand:]

  # Interaction terms: Multiply price with each encoded brand column
  for col in encoded_columns:
      df_encoded[f'PPL_{col}'] = df_encoded['PPL'] * df_encoded[col]

  # Columns used for modeling (including interaction terms)
  interaction_columns = [f'PPL_{col}' for col in encoded_columns]
  columns_taken = [item for sublist in [columns_in_use,encoded_columns, interaction_columns] for item in sublist]
  # data set used
  y_var=df_encoded['Volume']
  x_var=df_encoded[columns_in_use]
  # Separate the PPL column to generate cubic terms
  PPL = x_var[['PPL']].to_numpy()  # 'PPL' column

  # Generate cubic terms for PPL
  poly = PolynomialFeatures(degree=3, include_bias=False)  # Cubic terms for PPL
  PPL_poly = poly.fit_transform(PPL)  # Generate PPL, PPL^2, PPL^3

  # Remove PPL from x_var since we are adding its polynomial terms separately
  x_var = x_var.drop(columns=['PPL']).to_numpy()

  # Combine PPL polynomial terms with the rest of the linear features
  X = np.hstack((PPL_poly, x_var))
  # Standardize only the 'columns_in_use', leave encoded & interaction columns untouched
  scaler = StandardScaler()
  # Only scale columns_in_use
  x_var_scaled = scaler.fit_transform(X)

  # Concatenate the scaled columns with the one-hot and interaction columns
  x_full = np.concatenate([x_var_scaled, df_encoded[encoded_columns + interaction_columns].values], axis=1)
  # Convert target variable to numpy array
  Y = y_var.to_numpy()

  # X=np.c_[np.ones((X.shape[0], 1)), X]
  # scaler = StandardScaler()

  x_train, x_test, y_train, y_test = train_test_split(x_full, Y ,test_size=0.2, random_state=42)
  # x_train_custom = np.c_[np.ones((x_train.shape[0], 1)), x_train]
  x_train_custom_scaled = np.c_[np.ones((x_train.shape[0], 1)), x_train]
  x_test_custom_scaled = np.c_[np.ones((x_test.shape[0], 1)), x_test]




  # Prediction function (including intercept)
  def predict(X, W):
      return X.dot(W[1:]) + W[0]

  # Define the cost function (including L2 regularization)
  def cost_function(params, X, Y, l2_penalty):
      W = params
      Y_pred = predict(X, W)
      cost = np.sum((Y - Y_pred) ** 2) / X.shape[0] + l2_penalty * np.sum(W[1:] ** 2)
      return cost

  # MAPE calculation function
  def mean_absolute_percentage_error(y_true, y_pred):
      y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)
      weighted_mape=0
      for i in range(len(y_pred)):
        sub=abs(y_true[i]-y_pred[i])

        weighted_mape= weighted_mape+sub
      mape_weighted=weighted_mape/(np.sum(y_true))
      return mape_weighted*100

  def get_constraints(X, columns_in_use, encoded_columns):
      # This function returns constraints based on the correct indices of the features
      constraints = []

      # Tracking the index of each feature after combining polynomial terms
      index = 0

      # Polynomial terms for PPL (degree 3, so 3 terms: PPL, PPL^2, PPL^3)
      # Assuming you want to apply constraints to the linear PPL term (index 0) or adjust accordingly
      constraints.append({'type': 'ineq', 'fun': lambda params: -params[1] - 0.00000000000001})
      # constraints.append({'type': 'ineq', 'fun': lambda params: params[2] } )
      # constraints.append({'type': 'ineq', 'fun': lambda params: -params[3] - 0.0000000000001})

      # Increment index after the PPL polynomial terms (3 terms in total)
      index += 3

      # Constraints for other features (D1, PPL_Comp@Direct, etc.)
      for i, name in enumerate(columns_in_use[1:] + encoded_columns):  # Skipping 'PPL' since we already handled it
          if name in ['D1', 'PPL_Comp@Direct', 'PPL_cann']:
              constraints.append({'type': 'ineq', 'fun': lambda params, i=index: params[i]})
          elif name in ['D1_Comp@Direct', 'D1_cann']:
              constraints.append({'type': 'ineq', 'fun': lambda params, i=index: -params[i] - 0.00001})

          index += 1  # Move to the next feature index

      return constraints

  # Set regularization parameter (lambda for Ridge)
  # List of l2_penalty values to iterate over
  l2_penalty_list = [0.001,0.01, 0.1, 1, 10, 100]

  # Initialize variables to store the best results
  mape_custom_train_prev = float('inf')  # Start with a very large value for comparison
  best_l2_penalty = None
  best_W_opt = None
  best_y_pred_custom_train = None
  best_y_pred_custom_test = None

  for l2_penalty in l2_penalty_list:
      # Initial parameters (weights + intercept)
      initial_params = np.zeros(x_train_custom_scaled.shape[1] + 1)

      # Get the constraints
      constraints = get_constraints(X, columns_in_use, encoded_columns)

      # Minimize the cost function using SLSQP
      result = minimize(
          cost_function,
          initial_params,
          args=(x_train_custom_scaled, y_train, l2_penalty),
          constraints=constraints,
          method='SLSQP'
      )

      # Extract optimized parameters
      W_opt = result.x

      # Make predictions on the training set
      y_pred_custom_train = predict(x_train_custom_scaled, W_opt)
      y_pred_custom_test = predict(x_test_custom_scaled, W_opt)

      # Calculate MAPE for the custom implementation
      mape_custom_train_new = mean_absolute_percentage_error(y_train, y_pred_custom_train)

      # Update the best values if the current MAPE is better
      if mape_custom_train_new < mape_custom_train_prev:
          mape_custom_train_prev = mape_custom_train_new
          best_l2_penalty = l2_penalty
          best_W_opt = W_opt
          best_y_pred_custom_train = y_pred_custom_train
          best_y_pred_custom_test = y_pred_custom_test
          mape_custom_train = mape_custom_train_new  # Update train MAPE
          mape_custom_test = mean_absolute_percentage_error(y_test, y_pred_custom_test)  # Update test MAPE

  # Now after the loop, use the best_W_opt to report the final results
  print(f'Best l2_penalty: {best_l2_penalty}')
  print(f'MAPE (Train): {mape_custom_train:.2f}%')
  print(f'MAPE (Test): {mape_custom_test:.2f}%')

  # Optionally, you can plot the results using the best test predictions
  plt.figure(figsize=(16, 6))
  x_value = [i + 1 for i in range(len(y_test))]
  plt.plot(x_value, y_test, label='Actual Values', marker='o')
  plt.plot(x_value, best_y_pred_custom_test, label='Predicted Values by Custom Code', marker='x')

  # Adding title and labels
  plt.title(f'Actual vs Predicted Values (Best l2_penalty={best_l2_penalty})')
  plt.xlabel('Index')
  plt.ylabel('Values')
  plt.xticks(rotation=90)
  plt.legend()
  plt.show()
  return mape_custom_train,mape_custom_test


In [ ]:
unique_channel=df.Channel.unique()
# unique_channel=['Supermarkets']
channel=[]
ppg_list=[]
unique_brand_list=[]
train_mape=[]
test_mape=[]
for ch in unique_channel:
  df_ch=df[df['Channel']==ch]

  unique_ppg=df_ch.PPG.unique()
  # unique_ppg=['Standard Multi']
  for ppg in unique_ppg:
    df_ch_ppg=df_ch[df_ch['PPG']==ppg]
    unique_brand=df_ch_ppg.Brand.unique()


    df_ch_ppg=df_ch_ppg[['Market', 'Channel', 'Region', 'Category', 'SubCategory', 'Brand',
          'Variant', 'PackType', 'PPG', 'PackSize', 'Year',
          'Month', 'Week', 'Date', 'SalesValue', 'Volume', 'VolumeUnits', 'D1',
          'PPU', 'PPL', 'Cat_Vol_all_Chn', 'Cat_Vol_in_Chn', 'Seasonality',
          'Trend', 'Cat_Sales_in_Chn', 'Volume_sum_cann', 'D1_cann',
          'D1_Comp@Direct', 'D1_Comp@Indirect', 'PPU_cann', 'PPU_Comp@Direct',
          'PPU_Comp@Indirect', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect',
          'Up_Down_Index', 'Cat_Pr_Index']]

    df_ch_ppg.dropna(inplace=True)
    df_ch_ppg.reset_index(drop=True,inplace=True)
    # df_ch_ppg=remove_outliers_iqr(df_ch_ppg)
    if len(df_ch_ppg)>30:


      print(f'enough data points( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')
      mape_train,mape_test=Custom_ridge_Non_linear(df_ch_ppg)
      channel.append(ch)
      ppg_list.append(ppg)
      train_mape.append(mape_train)
      test_mape.append(mape_test)
      unique_brand_list.append(unique_brand)

    else :
      print(f'not enough data points ( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# d=df_ch_ppg.dropna()
# d

In [ ]:
result_df=pd.DataFrame({'Channel':channel,'PPG':ppg_list,'Brands':unique_brand_list,'Train_MAPE':train_mape,'Test_MAPE':test_mape})

result_df['Train_MAPE']=result_df['Train_MAPE'].round(2)
result_df['Test_MAPE']=result_df['Test_MAPE'].round(2)
result_df

,Channel,PPG,Brands,Train_MAPE,Test_MAPE
0,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",25.23,30.56
1,Convenience,Standard Multi,"[BRANSTON, HEINZ Standard, Private Label Std]",15.38,18.83
2,Convenience,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",15.35,15.63
3,Convenience,Small Single,"[HEINZ Flavoured, HEINZ Standard, Private Labe...",17.50,21.84
4,Convenience,Small Multi,[HEINZ Standard],21.50,25.52
5,Convenience,Small SNAP POTS,[HEINZ Standard],10.48,14.53
6,Iceland,Small Multi,"[BRANSTON, HEINZ Standard]",18.18,14.37
7,Iceland,Small Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",59.20,88.24
8,Iceland,Standard Multi,"[BRANSTON, HEINZ Standard]",11.87,10.46
9,Iceland,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Re...",24.11,36.00


# Different functions of PPL

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize
import matplotlib.pyplot as plt

def Custom_ridge(df_ch_ppg, transform_function='linear'):
    # Original columns in use
    columns_in_use = ['PPL', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect', 'D1', 'D1_cann',
                      'D1_Comp@Direct', 'D1_Comp@Indirect', 'Seasonality',
                      'Trend', 'Up_Down_Index', 'Cat_Pr_Index']

    # Apply transformation to the 'PPL' column based on the selected function
    if transform_function == 'reciprocal':
        df_ch_ppg['PPL_transformed'] = 1 / df_ch_ppg['PPL']
    elif transform_function == 'log':
        df_ch_ppg['PPL_transformed'] = np.log(df_ch_ppg['PPL'] + 1)  # Add 1 to avoid log(0)
    elif transform_function == 'exp_neg':
        df_ch_ppg['PPL_transformed'] = np.exp(-df_ch_ppg['PPL'])
    else:  # Default to linear (no transformation)
        df_ch_ppg['PPL_transformed'] = df_ch_ppg['PPL']

    # Replace the original 'PPL' in columns_in_use with 'PPL_transformed'
    columns_in_use[columns_in_use.index('PPL')] = 'PPL_transformed'

    # One-hot encode the 'Brand' column
    df_encoded = pd.get_dummies(df_ch_ppg, columns=['Brand'], drop_first=False)
    df_encoded = df_encoded.replace({True: 1, False: 0})

    # Get unique brand and encoded column names
    unique_brand = df_ch_ppg['Brand'].unique()
    len_unique_brand = len(unique_brand)
    df_encoded_columns = list(df_encoded.columns)
    encoded_columns = df_encoded_columns[-len_unique_brand:]

    # Interaction terms: Multiply transformed PPL with each encoded brand column
    for col in encoded_columns:
        df_encoded[f'PPL_{col}'] = df_encoded['PPL_transformed'] * df_encoded[col]

    # Columns used for modeling (including interaction terms)
    interaction_columns = [f'PPL_{col}' for col in encoded_columns]
    columns_taken = [item for sublist in [columns_in_use, encoded_columns, interaction_columns] for item in sublist]

    # Target variable
    y_var = df_encoded['Volume']
    columns_in_use = columns_in_use

    # Standardize only the 'columns_in_use'
    scaler = StandardScaler()
    x_var = df_encoded[columns_in_use]
    x_var_scaled = scaler.fit_transform(x_var)

    # Concatenate the scaled columns with the one-hot and interaction columns
    x_full = np.concatenate([x_var_scaled, df_encoded[encoded_columns + interaction_columns].values], axis=1)

    # Train-test split
    x_train, x_test, y_train, y_test = train_test_split(x_full, y_var, test_size=0.2, random_state=42)

    # Adding intercept column to x_train and x_test
    x_train_custom_scaled = x_train
    x_test_custom_scaled = x_test
    # Prediction function (including intercept)
    def predict(X, W):
        return X.dot(W[1:]) + W[0]

    # Define the cost function (including L2 regularization)
    def cost_function(params, X, Y, l2_penalty):
        W = params
        Y_pred = predict(X, W)
        cost = np.sum((Y - Y_pred) ** 2) / X.shape[0] + l2_penalty * np.sum(W** 2)
        return cost

    # MAPE calculation function
    def mean_absolute_percentage_error(y_true, y_pred):
        y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)
        weighted_mape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)
        return weighted_mape * 100

    # Define constraints based on the problem statement
    def get_constraints(feature_names):
        constraints = [{'type': 'ineq', 'fun': lambda params: -params[1] - 0.01}]
        for i, name in enumerate(feature_names):
            if name in ['D1', 'PPL_Comp@Direct', 'PPL_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: params[i]})
            elif name in ['D1_Comp@Direct', 'D1_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: -params[i] - 0.00001})
        return constraints

    # List of L2 penalty values to iterate over
    l2_penalty_list = [0.01, 0.1, 1, 10, 100]

    # Initialize variables to store the best results
    mape_custom_train_prev = float('inf')
    best_l2_penalty = None
    best_W_opt = None
    best_y_pred_custom_train = None
    best_y_pred_custom_test = None

    for l2_penalty in l2_penalty_list:
        # Initial parameters (weights + intercept)
        initial_params = np.zeros(x_train_custom_scaled.shape[1] + 1)

        # Get the constraints
        constraints = get_constraints(columns_taken)

        # Minimize the cost function using SLSQP
        result = minimize(
            cost_function,
            initial_params,
            args=(x_train_custom_scaled, y_train, l2_penalty),

            method='SLSQP'
        )

        # Extract optimized parameters
        W_opt = result.x

        # Make predictions on the training set
        y_pred_custom_train = predict(x_train_custom_scaled, W_opt)
        y_pred_custom_test = predict(x_test_custom_scaled, W_opt)

        # Calculate MAPE for the custom implementation
        mape_custom_train_new = mean_absolute_percentage_error(y_train, y_pred_custom_train)

        # Update the best values if the current MAPE is better
        if mape_custom_train_new < mape_custom_train_prev:
            mape_custom_train_prev = mape_custom_train_new
            best_l2_penalty = l2_penalty
            best_W_opt = W_opt
            best_y_pred_custom_train = y_pred_custom_train
            best_y_pred_custom_test = y_pred_custom_test
            mape_custom_train = mape_custom_train_new  # Update train MAPE
            mape_custom_test = mean_absolute_percentage_error(y_test, y_pred_custom_test)  # Update test MAPE

    # Report the final results
    print(f'Best l2_penalty: {best_l2_penalty}')
    print(f'MAPE (Train): {mape_custom_train:.2f}%')
    print(f'MAPE (Test): {mape_custom_test:.2f}%')

    # # Plot results using the best test predictions
    # plt.figure(figsize=(16, 6))
    # x_value = [i + 1 for i in range(len(y_test))]
    # plt.plot(x_value, y_test, label='Actual Values', marker='o')
    # plt.plot(x_value, best_y_pred_custom_test, label='Predicted Values by Custom Code', marker='x')
    # plt.title(f'Actual vs Predicted Values (Best l2_penalty={best_l2_penalty})')
    # plt.xlabel('Index')
    # plt.ylabel('Values')
    # plt.xticks(rotation=90)
    # plt.legend()
    # plt.show()

    return mape_custom_train, mape_custom_test, best_W_opt


In [ ]:
unique_channel=df.Channel.unique()
# unique_channel=['Supermarkets']
channel=[]
ppg_list=[]
unique_brand_list=[]
train_mape=[]
test_mape=[]
for ch in unique_channel:
  df_ch=df[df['Channel']==ch]

  unique_ppg=df_ch.PPG.unique()
  # unique_ppg=['Standard Multi']
  for ppg in unique_ppg:
    df_ch_ppg=df_ch[df_ch['PPG']==ppg]
    unique_brand=df_ch_ppg.Brand.unique()


    df_ch_ppg=df_ch_ppg[['Market', 'Channel', 'Region', 'Category', 'SubCategory', 'Brand',
          'Variant', 'PackType', 'PPG', 'PackSize', 'Year',
          'Month', 'Week', 'Date', 'SalesValue', 'Volume', 'VolumeUnits', 'D1',
          'PPU', 'PPL', 'Cat_Vol_all_Chn', 'Cat_Vol_in_Chn', 'Seasonality',
          'Trend', 'Cat_Sales_in_Chn', 'Volume_sum_cann', 'D1_cann',
          'D1_Comp@Direct', 'D1_Comp@Indirect', 'PPU_cann', 'PPU_Comp@Direct',
          'PPU_Comp@Indirect', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect',
          'Up_Down_Index', 'Cat_Pr_Index']]

    df_ch_ppg.dropna(inplace=True)
    df_ch_ppg.reset_index(drop=True,inplace=True)
    # df_ch_ppg=remove_outliers_iqr(df_ch_ppg)
    if len(df_ch_ppg)>30:


      print(f'enough data points( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')
      mape_train,mape_test,opt_w=Custom_ridge(df_ch_ppg, transform_function='linear')
      print(len(opt_w))
      # price_coef_index=1
      # price_range=[0,10]

      channel.append(ch)
      ppg_list.append(ppg)
      train_mape.append(mape_train)
      test_mape.append(mape_test)
      unique_brand_list.append(unique_brand)

    else :
      print(f'not enough data points ( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')

enough data points( 255   ) for combination of Channel>>> Convenience and PPG >>> Others and available brands are ['BRANSTON' 'HEINZ Standard' 'Restofcategory'] 
Best l2_penalty: 0.01
MAPE (Train): 16.63%
MAPE (Test): 20.38%
19
enough data points( 492   ) for combination of Channel>>> Convenience and PPG >>> Standard Multi and available brands are ['BRANSTON' 'HEINZ Standard' 'Private Label Std'] 
Best l2_penalty: 0.01
MAPE (Train): 13.75%
MAPE (Test): 16.29%
19
enough data points( 817   ) for combination of Channel>>> Convenience and PPG >>> Standard Single and available brands are ['BRANSTON' 'HEINZ Flavoured' 'HEINZ Standard' 'Private Label Std'
 'Restofcategory'] 
Best l2_penalty: 0.01
MAPE (Train): 16.84%
MAPE (Test): 16.66%
23
enough data points( 475   ) for combination of Channel>>> Convenience and PPG >>> Small Single and available brands are ['HEINZ Flavoured' 'HEINZ Standard' 'Private Label Std' 'Restofcategory'] 
Best l2_penalty: 0.01
MAPE (Train): 14.42%
MAPE (Test): 16.66%

In [ ]:
print(opt_w)

[ 3.99678519e+03 -7.21164366e+03 -6.33856216e+03  1.79206724e-12
  1.93712177e+03 -1.57673952e+03 -1.62566810e+03 -3.53016705e-12
 -8.09620933e+03  1.66070371e+03  5.19515964e+03  1.78335759e+04
  5.07386044e+03  3.99531683e+03  9.53013691e+03]


In [ ]:
result_df=pd.DataFrame({'Channel':channel,'PPG':ppg_list,'Brands':unique_brand_list,'Train_MAPE':train_mape,'Test_MAPE':test_mape})

result_df['Train_MAPE']=result_df['Train_MAPE'].round(2)
result_df['Test_MAPE']=result_df['Test_MAPE'].round(2)
result_df

,Channel,PPG,Brands,Train_MAPE,Test_MAPE
0,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",16.63,20.38
1,Convenience,Standard Multi,"[BRANSTON, HEINZ Standard, Private Label Std]",13.75,16.29
2,Convenience,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",16.84,16.66
3,Convenience,Small Single,"[HEINZ Flavoured, HEINZ Standard, Private Labe...",14.42,16.66
4,Convenience,Small Multi,[HEINZ Standard],8.84,10.87
5,Convenience,Small SNAP POTS,[HEINZ Standard],8.43,10.24
6,Iceland,Small Multi,"[BRANSTON, HEINZ Standard]",19.06,13.43
7,Iceland,Small Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",25.64,30.77
8,Iceland,Standard Multi,"[BRANSTON, HEINZ Standard]",11.88,10.71
9,Iceland,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Re...",18.92,24.14


# Demand Curve Plotting :


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.optimize import minimize
import matplotlib.pyplot as plt

def Custom_ridge(df_ch_ppg, transform_function):
    # Original columns in use
    columns_in_use = ['PPL', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect', 'D1', 'D1_cann',
                      'D1_Comp@Direct', 'D1_Comp@Indirect', 'Seasonality',
                      'Trend', 'Up_Down_Index', 'Cat_Pr_Index']

    # Apply transformation to the 'PPL' column based on the selected function
    if transform_function == 'reciprocal':
        df_ch_ppg['PPL_transformed'] = 1 / df_ch_ppg['PPL']
    # elif transform_function == 'log':
    #     df_ch_ppg['PPL_transformed'] = np.log(df_ch_ppg['PPL'] + 1)  # Add 1 to avoid log(0)
    elif transform_function == 'exp_neg':
        df_ch_ppg['PPL_transformed'] = np.exp(-df_ch_ppg['PPL'])
    else:  # Default to linear (no transformation)
        df_ch_ppg['PPL_transformed'] = df_ch_ppg['PPL']

    # Replace the original 'PPL' in columns_in_use with 'PPL_transformed'
    columns_in_use[columns_in_use.index('PPL')] = 'PPL_transformed'

    # One-hot encode the 'Brand' column
    df_encoded = pd.get_dummies(df_ch_ppg, columns=['Brand'], drop_first=False)
    df_encoded = df_encoded.replace({True: 1, False: 0})

    # Get unique brand and encoded column names
    unique_brand = df_ch_ppg['Brand'].unique()
    len_unique_brand = len(unique_brand)
    df_encoded_columns = list(df_encoded.columns)
    encoded_columns = df_encoded_columns[-len_unique_brand:]

    # Interaction terms: Multiply transformed PPL with each encoded brand column
    for col in encoded_columns:
        df_encoded[f'PPL_{col}'] = df_encoded['PPL_transformed'] * df_encoded[col]

    # Columns used for modeling (including interaction terms)
    interaction_columns = [f'PPL_{col}' for col in encoded_columns]
    columns_taken = [item for sublist in [columns_in_use, encoded_columns, interaction_columns] for item in sublist]

    # Target variable
    y_var = df_encoded['Volume']
    columns_in_use = columns_in_use

    # Standardize only the 'columns_in_use'
    scaler = StandardScaler()
    x_var = df_encoded[columns_in_use]
    x_var_scaled = scaler.fit_transform(x_var)

    # Concatenate the scaled columns with the one-hot and interaction columns
    x_full = np.concatenate([x_var_scaled, df_encoded[encoded_columns + interaction_columns].values], axis=1)
    print(x_full.shape,'>>>>>>>>>>>>>>>>>>>>>>>>>>>')
    # Train-test split
    x_train, x_test, y_train, y_test = train_test_split(x_full, y_var, test_size=0.2, random_state=42)

    # # Adding intercept column to x_train and x_test
    x_train_custom_scaled = x_train
    x_test_custom_scaled = x_test

    # Prediction function (including intercept)
    def predict(X, W):
        output = X.dot(W[1:]) + W[0]
        relu_output = np.maximum(0, output)  # Apply ReLU
        # print(relu_output)
        return relu_output

    # Define the cost function (including L2 regularization)
    def cost_function(params, X, Y, l2_penalty):
        W = params
        Y_pred = predict(X, W)
        cost = np.sum((Y - Y_pred) ** 2) / X.shape[0] + l2_penalty * np.sum(W[1:] ** 2)
        return cost

    # MAPE calculation function
    def mean_absolute_percentage_error(y_true, y_pred):
        y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)
        weighted_mape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)
        return weighted_mape * 100

    # Define constraints based on the problem statement
    def get_constraints(feature_names):
        constraints = [{'type': 'ineq', 'fun': lambda params: -params[1] - 0.01}]
        for i, name in enumerate(feature_names):
            if name in ['D1', 'PPL_Comp@Direct', 'PPL_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: params[i]})
            elif name in ['D1_Comp@Direct', 'D1_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: -params[i] - 0.00001})
        return constraints

    # List of L2 penalty values to iterate over
    l2_penalty_list = [0.01, 0.1, 1, 10, 100]

    # Initialize variables to store the best results
    mape_custom_train_prev = float('inf')
    best_l2_penalty = None
    best_W_opt = None
    best_y_pred_custom_train = None
    best_y_pred_custom_test = None

    for l2_penalty in l2_penalty_list:
        # Initial parameters (weights + intercept)
        initial_params = np.zeros(x_train_custom_scaled.shape[1] + 1)

        # Get the constraints
        constraints = get_constraints(columns_taken)

        # Minimize the cost function using SLSQP
        result = minimize(
            cost_function,
            initial_params,
            args=(x_train_custom_scaled, y_train, l2_penalty),
            method='SLSQP'
        )

        # Extract optimized parameters
        W_opt = result.x

        # Make predictions on the training set
        y_pred_custom_train = predict(x_train_custom_scaled, W_opt)
        y_pred_custom_test = predict(x_test_custom_scaled, W_opt)

        # Calculate MAPE for the custom implementation
        mape_custom_train_new = mean_absolute_percentage_error(y_train, y_pred_custom_train)

        # Update the best values if the current MAPE is better
        if mape_custom_train_new < mape_custom_train_prev:
            mape_custom_train_prev = mape_custom_train_new
            best_l2_penalty = l2_penalty
            best_W_opt = W_opt
            best_y_pred_custom_train = y_pred_custom_train
            best_y_pred_custom_test = y_pred_custom_test
            mape_custom_train = mape_custom_train_new  # Update train MAPE
            mape_custom_test = mean_absolute_percentage_error(y_test, y_pred_custom_test)  # Update test MAPE

    # Report the final results
    print(f'Best l2_penalty: {best_l2_penalty}')
    print(f'MAPE (Train): {mape_custom_train:.2f}%')
    print(f'MAPE (Test): {mape_custom_test:.2f}%')
    # Function to plot the demand curve
    # demand_predicted = x_full.dot(best_W_opt)
    print(transform_function,'Tranformation')
    def plot_demand_curve(x_full, best_W_opt, transform_function, scaler):
        # Generate a range of PPL values to simulate different demand scenarios
        if transform_function == 'reciprocal':
            PPL_values = np.linspace(0.1, 2.0, 100)  # Avoid zero to prevent division errors
            PPL_transformed = 1 / PPL_values

        elif transform_function == 'exp_neg':
            PPL_values = np.linspace(0, 10, 100)
            PPL_transformed = np.exp(-PPL_values)
        else:  # Default linear transformation
            PPL_values = np.linspace(0, 10, 100)
            PPL_transformed = PPL_values

        # Get weight for PPL (price-related term)
        W_opt_demand_curve = best_W_opt[1:].copy()
        W_opt_demand_curve[0] = 0  # Ignore intercept for demand calculation with PPL effect

        # Predict demand as a function of price
        mean_all = np.mean(x_full.dot(W_opt_demand_curve))
        demand = best_W_opt[0] + mean_all +  PPL_transformed* best_W_opt[1]

        # Clamp demand to ensure it is non-negative
        demand = np.maximum(demand, 0)

        # Calculate the price at which demand is zero
        zero_demand_price = -(best_W_opt[0]+mean_all) / best_W_opt[1] if best_W_opt[1] != 0 else None

        # Plot demand curve
        plt.plot( demand,PPL_values, label=f'Demand Curve ({transform_function} transformation)')
        plt.xlabel('Predicted Demand (Volume)')
        plt.ylabel('Price (PPL)')
        plt.legend()
        plt.title('Demand Curve')

        # # Mark the zero-demand price point on the graph, if applicable
        # if zero_demand_price and zero_demand_price > 0:
        #     plt.axvline(x=zero_demand_price, color='red', linestyle='--', label=f'Demand=0 at PPL={zero_demand_price:.2f}')
        #     plt.legend()

        # plt.show()
        # print(f"Price at which demand reaches zero: {np.abs(zero_demand_price)}")
        # return zero_demand_price
    # Call plot_demand_curve with modified parameters
    zero_demand_price=plot_demand_curve(x_full, best_W_opt, transform_function, scaler)
    # Optionally, you can plot the results using the best test predictions
    plt.figure(figsize=(16, 6))
    x_value = [i + 1 for i in range(len(y_test))]
    plt.plot(x_value, y_test, label='Actual Values', marker='o')
    plt.plot(x_value, best_y_pred_custom_test, label='Predicted Values by Custom Code', marker='x')

    # Adding title and labels
    plt.title(f'Actual vs Predicted Values (Best l2_penalty={best_l2_penalty})')
    plt.xlabel('Index')
    plt.ylabel('Values')
    plt.xticks(rotation=90)
    plt.legend()
    plt.show()

    return mape_custom_train, mape_custom_test, best_W_opt,zero_demand_price


In [ ]:
unique_channel=df.Channel.unique()
# unique_channel=['Supermarkets']
channel=[]
ppg_list=[]
unique_brand_list=[]
train_mape=[]
test_mape=[]
zero_demand_list=[]
for ch in unique_channel:
  df_ch=df[df['Channel']==ch]

  unique_ppg=df_ch.PPG.unique()
  # unique_ppg=['Standard Multi']
  for ppg in unique_ppg:
    df_ch_ppg=df_ch[df_ch['PPG']==ppg]
    unique_brand=df_ch_ppg.Brand.unique()


    df_ch_ppg=df_ch_ppg[['Market', 'Channel', 'Region', 'Category', 'SubCategory', 'Brand',
          'Variant', 'PackType', 'PPG', 'PackSize', 'Year',
          'Month', 'Week', 'Date', 'SalesValue', 'Volume', 'VolumeUnits', 'D1',
          'PPU', 'PPL', 'Cat_Vol_all_Chn', 'Cat_Vol_in_Chn', 'Seasonality',
          'Trend', 'Cat_Sales_in_Chn', 'Volume_sum_cann', 'D1_cann',
          'D1_Comp@Direct', 'D1_Comp@Indirect', 'PPU_cann', 'PPU_Comp@Direct',
          'PPU_Comp@Indirect', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect',
          'Up_Down_Index', 'Cat_Pr_Index']]

    df_ch_ppg.dropna(inplace=True)
    df_ch_ppg.reset_index(drop=True,inplace=True)
    # df_ch_ppg=remove_outliers_iqr(df_ch_ppg)
    if len(df_ch_ppg)>30:


      print(f'enough data points( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')
      transform_function='linear'
      mape_train,mape_test,opt_w,zero_demand_price=Custom_ridge(df_ch_ppg, transform_function)
      # print(len(opt_w))
      # price_coef_index=1
      # price_range=[0,10]

      channel.append(ch)
      ppg_list.append(ppg)
      train_mape.append(mape_train)
      test_mape.append(mape_test)
      unique_brand_list.append(unique_brand)
      zero_demand_list.append(zero_demand_price)


    else :
      print(f'not enough data points ( {len(df_ch_ppg)}   ) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brand} ')

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
result_df=pd.DataFrame({'Channel':channel,'PPG':ppg_list,'Brands':unique_brand_list,'Train_MAPE':train_mape,'Test_MAPE':test_mape})

result_df['Train_MAPE']=result_df['Train_MAPE'].round(2)
result_df['Test_MAPE']=result_df['Test_MAPE'].round(2)
result_df

,Channel,PPG,Brands,Train_MAPE,Test_MAPE
0,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",14.82,17.84
1,Convenience,Standard Multi,"[BRANSTON, HEINZ Standard, Private Label Std]",12.63,14.76
2,Convenience,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",14.17,13.70
3,Convenience,Small Single,"[HEINZ Flavoured, HEINZ Standard, Private Labe...",11.84,12.55
4,Convenience,Small Multi,[HEINZ Standard],8.83,10.98
5,Convenience,Small SNAP POTS,[HEINZ Standard],8.43,10.30
6,Iceland,Small Multi,"[BRANSTON, HEINZ Standard]",16.00,11.06
7,Iceland,Small Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",15.92,17.06
8,Iceland,Standard Multi,"[BRANSTON, HEINZ Standard]",11.92,10.77
9,Iceland,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Re...",18.03,21.17


# Table of LInear

In [ ]:
from prettytable import PrettyTable

# Data for the table
data = [
    ["Channel", "PPG", "Product Brands", "Train_MAPE", "Test_MAPE"],
    ["Channel 1", "PPG 1", "['Brand A', 'Brand B', 'Other Brands']", 14.82, 17.84],
    ["Channel 1", "PPG 2", "['Brand A', 'Brand B', 'Private Label Std']", 12.63, 14.76],
    ["Channel 1", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Other Brands']", 14.17, 13.70],
    ["Channel 1", "PPG 4", "['Brand C', 'Brand B', 'Brand D', 'Other Brands']", 11.84, 12.55],
    ["Channel 1", "PPG 5", "['Brand B']", 8.83, 10.98],
    ["Channel 1", "PPG 6", "['Brand B']", 8.43, 10.30],
    ["Channel 2", "PPG 5", "['Brand A', 'Brand B']", 16.00, 11.06],
    ["Channel 2", "PPG 4", "['Brand A', 'Brand C', 'Brand B', 'Brand D']", 15.92, 17.06],
    ["Channel 2", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Other Brands']", 11.92, 10.77],
    ["Channel 2", "PPG 2", "['Brand A', 'Brand B']", 18.03, 21.17],
    ["Channel 2", "PPG 1", "['Brand B']", 7.38, 8.78],
    ["Channel 3", "PPG 5", "['Brand A', 'Brand B']", 10.20, 11.66],
    ["Channel 3", "PPG 4", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Brand E', 'Other Brands']", 13.30, 12.24],
    ["Channel 3", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Brand E', 'Other Brands']", 11.99, 11.04],
    ["Channel 3", "PPG 2", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Brand E', 'Other Brands']", 15.48, 16.04],
    ["Channel 3", "PPG 6", "['Brand C', 'Brand B']", 11.28, 11.47],
    ["Channel 3", "PPG 1", "['Brand B', 'Other Brands']", 14.64, 12.12],
    ["Channel 4", "PPG 4", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Other Brands']", 12.71, 13.87],
    ["Channel 4", "PPG 2", "['Brand A', 'Brand B', 'Brand D']", 10.47, 10.44],
    ["Channel 4", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Brand E', 'Other Brands']", 16.57, 15.59],
    ["Channel 4", "PPG 1", "['Brand B']", 12.42, 10.22],
    ["Channel 4", "PPG 5", "['Brand B']", 12.57, 17.09],
    ["Channel 4", "PPG 6", "['Brand B']", 16.16, 14.22],
]

# Create PrettyTable
table = PrettyTable()
table.field_names = data[0]  # Set headers

# Add the data rows to the table
for row in data[1:]:
    table.add_row(row)

# Print the table
print(table)


+-----------+-------+-------------------------------------------------------------------------+------------+-----------+
|  Channel  |  PPG  |                              Product Brands                             | Train_MAPE | Test_MAPE |
+-----------+-------+-------------------------------------------------------------------------+------------+-----------+
| Channel 1 | PPG 1 |                  ['Brand A', 'Brand B', 'Other Brands']                 |   14.82    |   17.84   |
| Channel 1 | PPG 2 |               ['Brand A', 'Brand B', 'Private Label Std']               |   12.63    |   14.76   |
| Channel 1 | PPG 3 |       ['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Other Brands']      |   14.17    |    13.7   |
| Channel 1 | PPG 4 |            ['Brand C', 'Brand B', 'Brand D', 'Other Brands']            |   11.84    |   12.55   |
| Channel 1 | PPG 5 |                               ['Brand B']                               |    8.83    |   10.98   |
| Channel 1 | PPG 6 |           

# Table For Neg Exponential

In [ ]:
from prettytable import PrettyTable

# Data for the table (converted from your provided data)
data = [
    ["Channel", "PPG", "Product Brands", "Train_MAPE", "Test_MAPE"],
    ["Channel 1", "PPG 1", "['Brand A', 'Brand B', 'Other Brands']", 15.53, 17.34],
    ["Channel 1", "PPG 2", "['Brand A', 'Brand B', 'Private Label Std']", 15.92, 19.44],
    ["Channel 1", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Private Label Std']", 15.20, 15.84],
    ["Channel 1", "PPG 4", "['Brand C', 'Brand B', 'Private Label Std']", 14.98, 14.85],
    ["Channel 1", "PPG 5", "['Brand B']", 9.94, 11.57],
    ["Channel 1", "PPG 6", "['Brand B']", 7.79, 9.98],
    ["Channel 2", "PPG 1", "['Brand A', 'Brand B']", 14.86, 10.35],
    ["Channel 2", "PPG 2", "['Brand A', 'Brand C', 'Brand B']", 18.54, 19.98],
    ["Channel 2", "PPG 3", "['Brand A', 'Brand B']", 11.67, 10.32],
    ["Channel 2", "PPG 4", "['Brand A', 'Brand C', 'Brand B']", 19.84, 22.17],
    ["Channel 2", "PPG 5", "['Brand B']", 7.47, 8.88],
    ["Channel 3", "PPG 1", "['Brand A', 'Brand B']", 11.90, 12.53],
    ["Channel 3", "PPG 2", "['Brand A', 'Brand C', 'Brand B']", 14.63, 13.44],
    ["Channel 3", "PPG 3", "['Brand A', 'Brand B', 'Private Label Std']", 12.98, 12.10],
    ["Channel 3", "PPG 4", "['Brand A', 'Brand C', 'Brand B', 'Private Label Std']", 17.79, 17.95],
    ["Channel 3", "PPG 5", "['Brand C', 'Brand B']", 11.78, 11.00],
    ["Channel 3", "PPG 6", "['Brand B']", 14.92, 11.95],
    ["Channel 4", "PPG 1", "['Brand A', 'Brand B']", 12.98, 13.52],
    ["Channel 4", "PPG 2", "['Brand A', 'Brand B']", 10.91, 10.79],
    ["Channel 4", "PPG 3", "['Brand A', 'Brand C', 'Brand B']", 19.59, 19.08],
    ["Channel 4", "PPG 4", "['Brand A', 'Brand B']", 12.14, 9.98],
    ["Channel 4", "PPG 5", "['Brand A', 'Brand B']", 13.32, 16.95],
    ["Channel 4", "PPG 6", "['Brand B']", 15.37, 13.17]
]

# Create PrettyTable
table = PrettyTable()
table.field_names = data[0]  # Set headers

# Add the data rows to the table
for row in data[1:]:
    table.add_row(row)

# Print the table
print(table)



+-----------+-------+--------------------------------------------------------+------------+-----------+
|  Channel  |  PPG  |                     Product Brands                     | Train_MAPE | Test_MAPE |
+-----------+-------+--------------------------------------------------------+------------+-----------+
| Channel 1 | PPG 1 |         ['Brand A', 'Brand B', 'Other Brands']         |   15.53    |   17.34   |
| Channel 1 | PPG 2 |      ['Brand A', 'Brand B', 'Private Label Std']       |   15.92    |   19.44   |
| Channel 1 | PPG 3 | ['Brand A', 'Brand C', 'Brand B', 'Private Label Std'] |    15.2    |   15.84   |
| Channel 1 | PPG 4 |      ['Brand C', 'Brand B', 'Private Label Std']       |   14.98    |   14.85   |
| Channel 1 | PPG 5 |                      ['Brand B']                       |    9.94    |   11.57   |
| Channel 1 | PPG 6 |                      ['Brand B']                       |    7.79    |    9.98   |
| Channel 2 | PPG 1 |                 ['Brand A', 'Brand B']    

# Table for Reciprocal:

In [ ]:
from prettytable import PrettyTable

# Anonymized data for the new table
data = [
    ["Channel", "PPG", "Product Brands", "Train_MAPE", "Test_MAPE"],
    ["Channel 1", "PPG 1", "['Brand A', 'Brand B', 'Other Brands']", 15.94, 18.65],
    ["Channel 1", "PPG 2", "['Brand A', 'Brand B', 'Brand C']", 16.02, 19.27],
    ["Channel 1", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Brand D']", 14.98, 15.71],
    ["Channel 1", "PPG 4", "['Brand B', 'Brand C', 'Brand D']", 14.45, 14.71],
    ["Channel 1", "PPG 5", "['Brand B']", 8.85, 10.89],
    ["Channel 1", "PPG 6", "['Brand B']", 8.17, 10.21],
    ["Channel 2", "PPG 5", "['Brand A', 'Brand B']", 14.51, 10.18],
    ["Channel 2", "PPG 4", "['Brand A', 'Brand C', 'Brand B', 'Brand D']", 18.31, 18.85],
    ["Channel 2", "PPG 2", "['Brand A', 'Brand B']", 11.23, 10.02],
    ["Channel 2", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Other Brands']", 19.50, 22.67],
    ["Channel 2", "PPG 6", "['Brand B']", 7.40, 8.82],
    ["Channel 3", "PPG 5", "['Brand A', 'Brand B']", 11.75, 12.42],
    ["Channel 3", "PPG 4", "['Brand A', 'Brand C', 'Brand B', 'Brand D']", 14.29, 13.10],
    ["Channel 3", "PPG 2", "['Brand A', 'Brand C', 'Brand B', 'Brand D']", 12.80, 12.12],
    ["Channel 3", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Other Brands']", 16.46, 16.43],
    ["Channel 3", "PPG 6", "['Brand C', 'Brand B']", 11.93, 11.05],
    ["Channel 3", "PPG 1", "['Brand B', 'Other Brands']", 14.92, 11.96],
    ["Channel 4", "PPG 4", "['Brand A', 'Brand C', 'Brand B', 'Brand D']", 12.88, 13.38],
    ["Channel 4", "PPG 2", "['Brand A', 'Brand B', 'Brand C']", 11.06, 11.04],
    ["Channel 4", "PPG 3", "['Brand A', 'Brand C', 'Brand B', 'Brand D', 'Other Brands']", 18.66, 18.73],
    ["Channel 4", "PPG 1", "['Brand B']", 12.14, 9.91],
    ["Channel 4", "PPG 5", "['Brand B']", 13.12, 16.91],
    ["Channel 4", "PPG 6", "['Brand B']", 15.63, 13.29],
]

# Create PrettyTable
table = PrettyTable()
table.field_names = data[0]  # Set headers

for row in data[1:]:
    table.add_row(row)

# Print the table
print(table)


+-----------+-------+--------------------------------------------------------------+------------+-----------+
|  Channel  |  PPG  |                        Product Brands                        | Train_MAPE | Test_MAPE |
+-----------+-------+--------------------------------------------------------------+------------+-----------+
| Channel 1 | PPG 1 |            ['Brand A', 'Brand B', 'Other Brands']            |   15.94    |   18.65   |
| Channel 1 | PPG 2 |              ['Brand A', 'Brand B', 'Brand C']               |   16.02    |   19.27   |
| Channel 1 | PPG 3 |         ['Brand A', 'Brand C', 'Brand B', 'Brand D']         |   14.98    |   15.71   |
| Channel 1 | PPG 4 |              ['Brand B', 'Brand C', 'Brand D']               |   14.45    |   14.71   |
| Channel 1 | PPG 5 |                         ['Brand B']                          |    8.85    |   10.89   |
| Channel 1 | PPG 6 |                         ['Brand B']                          |    8.17    |   10.21   |
| Channel 

# Tried new Updated Code for brand mape also


In [ ]:
def Custom_ridge(df_ch_ppg, transform_function='exp_neg'):
    # Original columns in use
    columns_in_use = ['PPL', 'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect', 'D1', 'D1_cann',
                      'D1_Comp@Direct', 'D1_Comp@Indirect', 'Seasonality',
                      'Trend', 'Up_Down_Index', 'Cat_Pr_Index']

    # Apply transformation to the 'PPL' column based on the selected function
    if transform_function == 'reciprocal':
        df_ch_ppg['PPL_transformed'] = 1 / df_ch_ppg['PPL']
    elif transform_function == 'log':
        df_ch_ppg['PPL_transformed'] = np.log(df_ch_ppg['PPL'] + 1)
    elif transform_function == 'exp_neg':
        df_ch_ppg['PPL_transformed'] = np.exp(-df_ch_ppg['PPL'])
    else:
        df_ch_ppg['PPL_transformed'] = df_ch_ppg['PPL']

    # Replace the original 'PPL' in columns_in_use with 'PPL_transformed'
    columns_in_use[columns_in_use.index('PPL')] = 'PPL_transformed'

    # One-hot encode the 'Brand' column
    df_encoded = pd.get_dummies(df_ch_ppg, columns=['Brand'], drop_first=False)
    df_encoded = df_encoded.replace({True: 1, False: 0})

    # Get unique brand and encoded column names
    unique_brands = df_ch_ppg['Brand'].unique()
    encoded_columns = [f'Brand_{brand}' for brand in unique_brands]

    # Interaction terms: Multiply transformed PPL with each encoded brand column
    for col in encoded_columns:
        df_encoded[f'PPL_{col}'] = df_encoded['PPL_transformed'] * df_encoded[col]

    # Columns used for modeling (including interaction terms)
    interaction_columns = [f'PPL_{col}' for col in encoded_columns]
    columns_in_use += interaction_columns

    # Target variable
    y_var = df_encoded['Volume']

    # Standardize only the 'columns_in_use'
    scaler = StandardScaler()
    x_var = df_encoded[columns_in_use]
    x_var_scaled = scaler.fit_transform(x_var)

    # Train-test split
    x_train, x_test, y_train, y_test = train_test_split(x_var_scaled, y_var, test_size=0.2, random_state=42)

    # # Prediction function
    # def predict(X, W):
    # #     print(X.dot(W[1:]) + W[0])
    #    return X.dot(W[1:]) + W[0]
    def predict(X, W):
        output = X.dot(W[1:]) + W[0]
        relu_output = np.maximum(0, output)  # Apply ReLU
        # print(relu_output)
        return relu_output

    # Define cost function with L2 regularization
    def cost_function(params, X, Y, l2_penalty):
        W = params
        Y_pred = predict(X, W)
        cost = np.sum((Y - Y_pred) ** 2) / X.shape[0] + l2_penalty * np.sum(W[1:] ** 2)
        return cost
    def percentage(y_true, y_pred):
        y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)
        percentage_error = (np.abs((y_true - y_pred) / y_true)) * 100
        return percentage_error
    # Calculate MAPE
    def mean_absolute_percentage_error(y_true, y_pred):
        y_true = np.where(y_true == 0, np.finfo(float).eps, y_true)

        weighted_mape = np.sum(percentage(y_true,y_pred)*y_true) / np.sum(y_true)
        return weighted_mape

    # Constraints for optimization
    def get_constraints(feature_names):
        constraints = [{'type': 'ineq', 'fun': lambda params: -params[1] - 0.01}]
        for i, name in enumerate(feature_names):
            if name in ['D1', 'PPL_Comp@Direct', 'PPL_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: params[i]})
            elif name in ['D1_Comp@Direct', 'D1_cann']:
                constraints.append({'type': 'ineq', 'fun': lambda params, i=i: -params[i] - 0.00001})
        return constraints


    # L2 penalty values
    l2_penalty_list = [0.01, 0.1, 1, 10, 100]
    best_mape, best_W_opt = float('inf'), None
    if transform_function == 'linear':
        for l2_penalty in l2_penalty_list:
          # Initial parameters (weights + intercept)
          initial_params = np.zeros(x_train.shape[1] + 1)

          # Get the constraints
          constraints = get_constraints(columns_in_use)

          # Minimize the cost function using SLSQP
          result = minimize(
              cost_function,
              initial_params,
              args=(x_train, y_train, l2_penalty),
              constraints=constraints,
              method='SLSQP'
          )

          # Extract optimized parameters
          W_opt = result.x
          y_pred_train = predict(x_train, W_opt)
          mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
          if mape_train < best_mape:
              best_mape, best_W_opt = mape_train, W_opt
    else:
        for l2_penalty in l2_penalty_list:
            initial_params = np.zeros(x_train.shape[1] + 1)
            constraints = get_constraints(columns_in_use)
            result = minimize(cost_function, initial_params, args=(x_train, y_train, l2_penalty), method='SLSQP')
            W_opt = result.x
            y_pred_train = predict(x_train, W_opt)
            mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
            if mape_train < best_mape:
                best_mape, best_W_opt = mape_train, W_opt

    # Calculate overall test MAPE
    y_pred_test = predict(x_test, best_W_opt)
    overall_mape_test = mean_absolute_percentage_error(y_test, y_pred_test)

    # Calculate MAPE for each brand
    brand_mapes = {}
    for brand, col_name in zip(unique_brands, encoded_columns):
        brand_indices = df_encoded[df_encoded[col_name] == 1].index
        x_brand = x_var_scaled[brand_indices]
        y_brand = y_var.iloc[brand_indices]
        y_pred_brand = predict(x_brand, best_W_opt)
        brand_mapes[brand] = mean_absolute_percentage_error(y_brand, y_pred_brand)

    # Print final results
    print(f'Best L2 Penalty: {l2_penalty_list[np.argmin(brand_mapes)]}')
    print(f'Overall MAPE (Test): {overall_mape_test:.2f}%')
    print("Brand-wise MAPE:", brand_mapes)

    return best_mape, overall_mape_test, best_W_opt, brand_mapes


In [ ]:
# Initialize lists for storing the expanded data
channel_list = []
ppg_list = []
brand_list = []
train_mape_list = []
test_mape_list = []
brand_list_unique = []
brand_mape_list=[]
index_list=[]
# Iterate over each unique channel
for ch in unique_channel:
    df_ch = df[df['Channel'] == ch]
    unique_ppg = df_ch.PPG.unique()

    # Iterate over each unique PPG within the current channel
    for ppg in unique_ppg:
        df_ch_ppg = df_ch[df_ch['PPG'] == ppg]
        unique_brands = df_ch_ppg.Brand.unique()

        # Filter relevant columns and clean up data
        df_ch_ppg = df_ch_ppg[['Market', 'Channel', 'Region', 'Category', 'SubCategory', 'Brand',
                               'Variant', 'PackType', 'PPG', 'PackSize', 'Year', 'Month', 'Week',
                               'Date', 'SalesValue', 'Volume', 'VolumeUnits', 'D1', 'PPU', 'PPL',
                               'Cat_Vol_all_Chn', 'Cat_Vol_in_Chn', 'Seasonality', 'Trend',
                               'Cat_Sales_in_Chn', 'Volume_sum_cann', 'D1_cann', 'D1_Comp@Direct',
                               'D1_Comp@Indirect', 'PPU_cann', 'PPU_Comp@Direct', 'PPU_Comp@Indirect',
                               'PPL_cann', 'PPL_Comp@Direct', 'PPL_Comp@Indirect', 'Up_Down_Index',
                               'Cat_Pr_Index']]

        df_ch_ppg.dropna(inplace=True)
        df_ch_ppg.reset_index(drop=True, inplace=True)

        # Only proceed if there are enough data points
        if len(df_ch_ppg) > 30:
            print(f'enough data points ({len(df_ch_ppg)}) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brands}')

            # Compute MAPE and zero demand price using the custom function
            list_mape_train=[]
            function_list=['linear','reciprocal', 'exp_neg']
            for trans in function_list:

              mape_train, mape_test, opt_w, brand_wise_mape = Custom_ridge(df_ch_ppg, transform_function=trans)
              list_mape_train.append([mape_train,mape_test,opt_w,brand_wise_mape])
              # print(brand_wise_mape)
            min_index = min(range(len(list_mape_train)), key=lambda i: list_mape_train[i][0])

            # Get the complete list with the minimum first element
            min_list = list_mape_train[min_index]
            brand_wise_mape=min_list[3]
            mape_train=min_list[0]
            mape_test=min_list[1]
            print(mape_train,mape_test)
            for key, value in brand_wise_mape.items():
                  brand_list_unique.append(key)
                  brand_mape_list.append(value)
            # Store values for each brand

                  channel_list.append(ch)
                  ppg_list.append(ppg)
                  brand_list.append(unique_brands)
                  train_mape_list.append(mape_train)  # Get train MAPE for brand
                  test_mape_list.append(mape_test)
                  index_list.append(function_list[min_index])# Get test MAPE for brand


        else:
              print(f'not enough data points ({len(df_ch_ppg)}) for combination of Channel>>> {ch} and PPG >>> {ppg} and available brands are {unique_brands}')





enough data points (255) for combination of Channel>>> Convenience and PPG >>> Others and available brands are ['BRANSTON' 'HEINZ Standard' 'Restofcategory']
Best L2 Penalty: 0.01
Overall MAPE (Test): 55.96%
Brand-wise MAPE: {'BRANSTON': 41.640899462294776, 'HEINZ Standard': 38.594675784854495, 'Restofcategory': 151.75229054258523}
Best L2 Penalty: 0.01
Overall MAPE (Test): 19.52%
Brand-wise MAPE: {'BRANSTON': 27.74555777900282, 'HEINZ Standard': 12.689258947017608, 'Restofcategory': 48.34192912776584}
Best L2 Penalty: 0.01
Overall MAPE (Test): 18.94%
Brand-wise MAPE: {'BRANSTON': 26.665436189993653, 'HEINZ Standard': 12.7900070011735, 'Restofcategory': 43.335309379205896}
16.10299005083289 18.93769819265408
enough data points (492) for combination of Channel>>> Convenience and PPG >>> Standard Multi and available brands are ['BRANSTON' 'HEINZ Standard' 'Private Label Std']
Best L2 Penalty: 0.01
Overall MAPE (Test): 27.99%
Brand-wise MAPE: {'BRANSTON': 32.7152864729484, 'HEINZ Standard

In [ ]:
# Create the final DataFrame with the required columns
result_df = pd.DataFrame({
    'Channel': channel_list,
    'PPG': ppg_list,
    'Unique_Brands': brand_list,
    'Train_MAPE': train_mape_list,
    'Test_MAPE': test_mape_list,
    # 'Brand': brand_list_unique,
    # 'Brand_mape': brand_mape_list,
    'Best_transformation':index_list
})

# Round MAPE columns to two decimal places
result_df['Train_MAPE'] = result_df['Train_MAPE'].round(2)
result_df['Test_MAPE'] = result_df['Test_MAPE'].round(2)
# result_df['Brand_mape'] = result_df['Brand_mape'].round(2)

In [ ]:
result_df

,Channel,PPG,Unique_Brands,Train_MAPE,Test_MAPE,Best_transformation
0,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",16.10,18.94,exp_neg
1,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",16.10,18.94,exp_neg
2,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",16.10,18.94,exp_neg
3,Convenience,Standard Multi,"[BRANSTON, HEINZ Standard, Private Label Std]",12.80,15.67,exp_neg
4,Convenience,Standard Multi,"[BRANSTON, HEINZ Standard, Private Label Std]",12.80,15.67,exp_neg
...,...,...,...,...,...,...
66,Tesco,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",13.38,12.68,exp_neg
67,Tesco,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",13.38,12.68,exp_neg
68,Tesco,Others,[HEINZ Standard],12.14,9.96,exp_neg
69,Tesco,Small Multi,[HEINZ Standard],13.14,16.98,reciprocal


In [ ]:
result_df['Unique_Brands'] = result_df['Unique_Brands'].apply(tuple)  # Convert lists to tuples
result_df.drop_duplicates(inplace=True)
result_df['Unique_Brands'] = result_df['Unique_Brands'].apply(list)  # Convert tuples back to lists if needed
result_df

,Channel,PPG,Unique_Brands,Train_MAPE,Test_MAPE,Best_transformation
0,Convenience,Others,"[BRANSTON, HEINZ Standard, Restofcategory]",16.10,18.94,exp_neg
3,Convenience,Standard Multi,"[BRANSTON, HEINZ Standard, Private Label Std]",12.80,15.67,exp_neg
6,Convenience,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",13.92,15.16,reciprocal
11,Convenience,Small Single,"[HEINZ Flavoured, HEINZ Standard, Private Labe...",11.01,11.24,reciprocal
15,Convenience,Small Multi,[HEINZ Standard],8.83,10.94,reciprocal
16,Convenience,Small SNAP POTS,[HEINZ Standard],7.79,10.05,exp_neg
17,Iceland,Small Multi,"[BRANSTON, HEINZ Standard]",13.87,10.06,exp_neg
19,Iceland,Small Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Pr...",18.28,18.56,reciprocal
23,Iceland,Standard Multi,"[BRANSTON, HEINZ Standard]",11.42,10.09,reciprocal
25,Iceland,Standard Single,"[BRANSTON, HEINZ Flavoured, HEINZ Standard, Re...",13.08,18.39,exp_neg


In [ ]:
from prettytable import PrettyTable

# Data (from your provided data)
data = [
    ["Convenience", "Others", ["BRANSTON", "HEINZ Standard", "Restofcategory"], 16.10, 18.94, "exp_neg"],
    ["Convenience", "Standard Multi", ["BRANSTON", "HEINZ Standard", "Private Label Std"], 12.80, 15.67, "exp_neg"],
    ["Convenience", "Standard Single", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Private Label Small"], 13.92, 15.16, "reciprocal"],
    ["Convenience", "Small Single", ["HEINZ Flavoured", "HEINZ Standard", "Private Label Small"], 11.01, 11.24, "reciprocal"],
    ["Convenience", "Small Multi", ["HEINZ Standard"], 8.83, 10.94, "reciprocal"],
    ["Convenience", "Small SNAP POTS", ["HEINZ Standard"], 7.79, 10.05, "exp_neg"],
    ["Iceland", "Small Multi", ["BRANSTON", "HEINZ Standard"], 13.87, 10.06, "exp_neg"],
    ["Iceland", "Small Single", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Private Label Small"], 18.28, 18.56, "reciprocal"],
    ["Iceland", "Standard Multi", ["BRANSTON", "HEINZ Standard"], 11.42, 10.09, "reciprocal"],
    ["Iceland", "Standard Single", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Restofcategory"], 13.08, 18.39, "exp_neg"],
    ["Iceland", "Small SNAP POTS", ["HEINZ Standard"], 7.40, 8.81, "reciprocal"],
    ["Supermarkets", "Small Multi", ["BRANSTON", "HEINZ Standard"], 9.98, 10.91, "reciprocal"],
    ["Supermarkets", "Small Single", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Private Label Small"], 11.18, 10.16, "reciprocal"],
    ["Supermarkets", "Standard Multi", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Private Label Std"], 12.81, 11.79, "reciprocal"],
    ["Supermarkets", "Standard Single", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Private Label Std"], 11.66, 12.56, "exp_neg"],
    ["Supermarkets", "Small SNAP POTS", ["HEINZ Flavoured", "HEINZ Standard"], 10.99, 10.66, "exp_neg"],
    ["Supermarkets", "Others", ["HEINZ Standard", "Restofcategory"], 14.02, 12.26, "exp_neg"],
    ["Tesco", "Small Single", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Private Label Small"], 10.82, 12.10, "reciprocal"],
    ["Tesco", "Standard Multi", ["BRANSTON", "HEINZ Standard", "Private Label Std"], 10.94, 10.83, "exp_neg"],
    ["Tesco", "Standard Single", ["BRANSTON", "HEINZ Flavoured", "HEINZ Standard", "Private Label Small"], 13.38, 12.68, "exp_neg"],
    ["Tesco", "Others", ["HEINZ Standard"], 12.14, 9.96, "exp_neg"],
    ["Tesco", "Small Multi", ["HEINZ Standard"], 13.14, 16.98, "reciprocal"],
    ["Tesco", "Small SNAP POTS", ["HEINZ Standard"], 15.37, 13.24, "exp_neg"]
]

# Mappings
channel_mapping = {
    "Convenience": "Channel 1",
    "Iceland": "Channel 2",
    "Supermarkets": "Channel 3",
    "Tesco": "Channel 4"
}

ppg_mapping = {
    "Small Multi": "PPG 1",
    "Small Single": "PPG 2",
    "Standard Multi": "PPG 3",
    "Standard Single": "PPG 4",
    "Small SNAP POTS": "PPG 5",
    "Others": "PPG 6"
}

brand_mapping = {
    "BRANSTON": "Brand A",
    "HEINZ Standard": "Brand B",
    "HEINZ Flavoured": "Brand C",
    "Private Label Std": "Brand D",
    "Private Label Small": "Brand E",
    "Restofcategory": "Other Brands"
}

# Create PrettyTable
table = PrettyTable()

# Add headers
table.field_names = ["Channel", "PPG", "Unique Brands", "Train MAPE", "Test MAPE", "Best Transformation"]

# Add rows to the table
for row in data:
    channel = channel_mapping.get(row[0], row[0])
    ppg = ppg_mapping.get(row[1], row[1])
    brands = [brand_mapping.get(brand, brand) for brand in row[2]]
    train_mape = row[3]
    test_mape = row[4]
    best_transformation = row[5]

    # Add the transformed row to the table
    table.add_row([channel, ppg, ', '.join(brands), train_mape, test_mape, best_transformation])

# Print the table
print(table)


+-----------+-------+-----------------------------------------+------------+-----------+---------------------+
|  Channel  |  PPG  |              Unique Brands              | Train MAPE | Test MAPE | Best Transformation |
+-----------+-------+-----------------------------------------+------------+-----------+---------------------+
| Channel 1 | PPG 6 |      Brand A, Brand B, Other Brands     |    16.1    |   18.94   |       exp_neg       |
| Channel 1 | PPG 3 |        Brand A, Brand B, Brand D        |    12.8    |   15.67   |       exp_neg       |
| Channel 1 | PPG 4 |    Brand A, Brand C, Brand B, Brand E   |   13.92    |   15.16   |      reciprocal     |
| Channel 1 | PPG 2 |        Brand C, Brand B, Brand E        |   11.01    |   11.24   |      reciprocal     |
| Channel 1 | PPG 1 |                 Brand B                 |    8.83    |   10.94   |      reciprocal     |
| Channel 1 | PPG 5 |                 Brand B                 |    7.79    |   10.05   |       exp_neg       |
|

Channel Mapping:
Channel 1 → "Convenience"
Channel 2 → "Iceland"
Channel 3 → "Supermarkets"
Channel 4 → "Tesco"
PPG Mapping:
PPG 1 → "Small Multi"
PPG 2 → "Small Single"
PPG 3 → "Standard Multi"
PPG 4 → "Standard Single"
PPG 5 → "Small SNAP POTS"
PPG 6 → "Others"
Brand Mapping:
Brand A → "BRANSTON"
Brand B → "HEINZ Standard"
Brand C → "HEINZ Flavoured"
Brand D → "Private Label Std"
Brand E → "Private Label Small"
Other Brands → "Restofcategory"